# Chapter 16: Projective Conics

Source span: printed Volume II pages 170-217; PDF pages 179-226.

This notebook treats a projective conic as a quadratic equation in homogeneous coordinates and as a copy of the projective line disguised inside the plane. The central question is practical: when a curve is defined only by incidence, tangency, projection, and intersection data, which quantities survive a change of projective coordinates?

The answer is that a nondegenerate conic is rigid enough to carry one-dimensional projective geometry. A rational parameter on the conic behaves like a coordinate on `P^1`; cross-ratios are read from that parameter; projectivities of the plane act by fractional linear changes on the parameter; and incidence theorems such as Pascal become numerical tests of collinearity. The same quadratic-matrix representation also turns tangents, polars, intersections, and pencils into concrete linear algebra.

Every diagram below is generated from code. The goal is not to make a gallery, but to make the algebra visible: each saved figure has an accompanying identity or residual check.

## Translation guide

Projective conic language becomes computational language as follows.

- A point in the projective plane is a nonzero vector `X = [x, y, z]`, with scalar multiples representing the same point. The affine chart `z = 1` displays it as `(x/z, y/z)` when `z` is not zero.
- A line is a vector `l = [a, b, c]`, and incidence is the dot product `l @ X = 0`. The line through two points is `cross(P, Q)`; the intersection of two lines is `cross(l, m)`.
- A conic is a symmetric matrix `Q`, with equation `X.T @ Q @ X = 0`. The unit circle model uses `diag(1, 1, -1)`, but projective geometry does not privilege circles over any other nondegenerate conic.
- A projective transform `H` sends points to `H @ X`. The conic matrix transforms to `inv(H).T @ Q @ inv(H)` so that the equation remains true after moving the points.
- A tangent at a point `P` on the conic is the line `Q @ P`. For an off-conic point `S`, the line `Q @ S` is its polar. This is the computational form of the polarity attached to a conic.
- A rational parameter `u` on the standard conic is implemented by `[(u*u - 1), 2*u, (u*u + 1)]`. The missing parameter value at infinity gives `[1, 0, 1]`. This identifies the conic with `P^1`.
- The cross-ratio of four conic points is computed from their `P^1` parameters. Any fractional linear reparameterization preserves it.
- A pencil of conics is a one-parameter family `Q0 + lambda Q1`. Singular members occur where the determinant vanishes; visually these are the places where the conic breaks into lines or loses rank.

## Visualization storyboard

The notebook builds six linked views.

1. A rational parametrization diagram shows a conic as a projective line wrapped into the plane.
2. A cross-ratio display compares four parameters before and after a fractional linear change.
3. A Pascal diagram places a hexagon on a conic and checks that the three opposite-side intersections are collinear.
4. A polarity diagram shows an exterior point, its polar chord, and the two tangent lines it controls.
5. A Bezout example solves two conics whose four intersections are all real and visible in one affine chart.
6. A pencil explorer stores an interactive Plotly HTML view of `Q0 + lambda Q1`, paired with determinant data that identifies singular parameters.

The final cell gathers the numerical residuals and artifact paths into `artifacts/chapter-16/checks/final_sanity.json`.

In [ ]:
from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Geometry II book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_csv, save_json, save_plotly_html
from utils.geometry import conic_matrix, cross_ratio, dehomogeneous, polar_line
from utils.plotting import COLORS, new_axes, plot_line, plot_points, save_figure

CHAPTER = 16
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-16"
for kind in ["figures", "plots", "tables", "checks"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

def artifact(kind: str, filename: str) -> Path:
    return ARTIFACT_ROOT / kind / filename

def notebook_relative(path: Path) -> Path:
    return Path("..") / Path(path).resolve().relative_to(BOOK_ROOT)

np.set_printoptions(precision=6, suppress=True)
BOOK_ROOT.relative_to(BOOK_ROOT.parent)

## Conics as parametrized projective lines

The standard conic `x^2 + y^2 - z^2 = 0` is a circle only after choosing the affine chart `z = 1`. Projectively, its more important feature is that it can be swept by one parameter. The formula

`P(u) = [u^2 - 1, 2u, u^2 + 1]`

satisfies the quadratic equation for every finite `u`. The point at parameter infinity is `[1, 0, 1]`, completing the copy of `P^1`. A computational advantage of this formula is that many projective statements about conics become statements about a single scalar parameter.

The first figure marks several parameter values and draws chords from the point at infinity in the parameter line. These chords are not the definition of the conic, but they remind us where the parametrization comes from: intersecting a variable line through a fixed base point with the conic gives the second intersection point.

In [ ]:
Q_circle = conic_matrix("unit_circle")

def hpoint_on_conic(u: float) -> np.ndarray:
    if np.isinf(u):
        return np.array([1.0, 0.0, 1.0])
    u = float(u)
    return np.array([u * u - 1.0, 2.0 * u, u * u + 1.0])

def hnormalize(point: np.ndarray) -> np.ndarray:
    point = np.asarray(point, dtype=float)
    scale = point[-1] if abs(point[-1]) > 1e-12 else np.linalg.norm(point)
    return point / scale

def hline(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    return np.cross(p, q)

def hintersection(l: np.ndarray, m: np.ndarray) -> np.ndarray:
    return np.cross(l, m)

def hconic_value(Q: np.ndarray, point: np.ndarray) -> float:
    point = np.asarray(point, dtype=float)
    return float(point @ Q @ point)

def affine(point: np.ndarray) -> np.ndarray:
    return dehomogeneous(np.asarray(point, dtype=float))

u_grid = np.linspace(-8.0, 8.0, 700)
curve_h = np.array([hpoint_on_conic(u) for u in u_grid])
curve = np.array([affine(p) for p in curve_h])
sample_u = np.array([-3.0, -1.2, -0.35, 0.6, 1.8, 4.0])
sample_h = np.array([hpoint_on_conic(u) for u in sample_u])
sample = np.array([affine(p) for p in sample_h])
base = affine(hpoint_on_conic(np.inf))

fig, ax = new_axes(figsize=(7.4, 6.2), title="A rational parameter wraps P1 around a conic")
ax.plot(curve[:, 0], curve[:, 1], color=COLORS["ink"], linewidth=2.2, label="standard conic")
for point, u in zip(sample, sample_u):
    ax.plot([base[0], point[0]], [base[1], point[1]], color=COLORS["gray"], linewidth=1.0, alpha=0.55)
plot_points(ax, sample, labels=[f"u={u:g}" for u in sample_u], color=COLORS["blue"], size=48)
ax.scatter([base[0]], [base[1]], s=70, c=COLORS["orange"], edgecolor="white", linewidth=0.9, zorder=6)
ax.text(base[0] + 0.05, base[1] - 0.1, "u=infinity", fontsize=8, color=COLORS["orange"])
ax.set_xlim(-1.55, 1.55)
ax.set_ylim(-1.35, 1.35)
ax.legend(loc="lower left", fontsize=8)
param_fig = save_figure(fig, artifact("figures", "parametrized_conic_atlas.png"))
param_residual = max(abs(hconic_value(Q_circle, p)) for p in sample_h)
display_artifact(notebook_relative(param_fig), width=740)

## Cross-ratio on a conic

A line has a cross-ratio because it is one-dimensional projective space. Once a conic is parametrized by `P^1`, four points on the conic inherit the same invariant. This is more than a convenient coordinate trick. A projectivity preserving the conic acts on the parameter by a fractional linear transformation, so the cross-ratio of four conic points is independent of the chosen projective coordinates.

The next cell uses a numerical fractional linear map. The two rows in the table have different parameter values and different displayed points, but the cross-ratio agrees up to floating point error. This is the invariant that later lets conic geometry behave like projective line geometry.

In [ ]:
def mobius(u: np.ndarray, a: float, b: float, c: float, d: float) -> np.ndarray:
    u = np.asarray(u, dtype=float)
    return (a * u + b) / (c * u + d)

cr_params = np.array([-2.2, -0.45, 0.75, 2.4])
M = (1.25, -0.35, 0.22, 1.10)
cr_mapped_params = mobius(cr_params, *M)
cr_original = cross_ratio(*cr_params)
cr_mapped = cross_ratio(*cr_mapped_params)
cross_ratio_error = abs(cr_original - cr_mapped)

rows = []
for label, params, value in [
    ("original", cr_params, cr_original),
    ("after_mobius", cr_mapped_params, cr_mapped),
]:
    row = {"label": label, "cross_ratio": float(value)}
    for idx, u in enumerate(params, start=1):
        row[f"u{idx}"] = float(u)
    rows.append(row)
cr_table = save_csv(rows, artifact("tables", "cross_ratio_samples.csv"))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.8), constrained_layout=True)
for ax, title, params, color in [
    (axes[0], "original parameters", cr_params, COLORS["blue"]),
    (axes[1], "after fractional linear change", cr_mapped_params, COLORS["teal"]),
]:
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.set_facecolor("white")
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold")
    ax.plot(curve[:, 0], curve[:, 1], color=COLORS["ink"], linewidth=1.8)
    pts = np.array([affine(hpoint_on_conic(u)) for u in params])
    ax.scatter(pts[:, 0], pts[:, 1], s=58, c=color, edgecolor="white", linewidth=0.9, zorder=5)
    for point, name, u in zip(pts, list("ABCD"), params):
        ax.text(point[0] + 0.04, point[1] + 0.04, f"{name}\nu={u:.2f}", fontsize=8)
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.25, 1.25)
    for spine in ax.spines.values():
        spine.set_color("#d1d5db")
fig.suptitle(f"Cross-ratio: {cr_original:.6f} vs {cr_mapped:.6f}", fontsize=13)
cr_fig = save_figure(fig, artifact("figures", "cross_ratio_projection.png"))

display(pd.DataFrame(rows))
display_artifact(notebook_relative(cr_fig), width=900)

## Pascal as a collinearity computation

Pascal's theorem says that if six points lie on a nondegenerate conic, then the three intersections of opposite sides of the hexagon are collinear. In homogeneous coordinates this is almost embarrassingly direct. Lines are cross products of adjacent vertices, intersections are cross products of lines, and collinearity is a single determinant or, equivalently, incidence with the line through two of the intersection points.

The diagram below uses six parameter values on the standard conic. The hexagon is not chosen to look regular; regular-looking choices often hide special symmetries or send a Pascal point to infinity. The residual in `pascal_residual` is the normalized incidence error for the third Pascal point on the line through the first two.

In [ ]:
pascal_u = np.array([-4.0, -1.5, -0.2, 0.7, 1.7, 4.0])
labels = list("ABCDEF")
P_h = [hpoint_on_conic(u) for u in pascal_u]
P = np.array([affine(p) for p in P_h])

side_lines = {
    "AB": hline(P_h[0], P_h[1]),
    "BC": hline(P_h[1], P_h[2]),
    "CD": hline(P_h[2], P_h[3]),
    "DE": hline(P_h[3], P_h[4]),
    "EF": hline(P_h[4], P_h[5]),
    "FA": hline(P_h[5], P_h[0]),
}
X1_h = hintersection(side_lines["AB"], side_lines["DE"])
X2_h = hintersection(side_lines["BC"], side_lines["EF"])
X3_h = hintersection(side_lines["CD"], side_lines["FA"])
pascal_line = hline(X1_h, X2_h)
pascal_residual = abs(float(pascal_line @ X3_h)) / (np.linalg.norm(pascal_line) * np.linalg.norm(X3_h))
X = np.array([affine(x) for x in [X1_h, X2_h, X3_h]])

fig, ax = new_axes(figsize=(8.0, 6.5), title="Pascal line from a hexagon on a conic")
ax.plot(curve[:, 0], curve[:, 1], color=COLORS["ink"], linewidth=2.0, label="conic")
hexagon = np.vstack([P, P[0]])
ax.plot(hexagon[:, 0], hexagon[:, 1], color=COLORS["blue"], linewidth=1.8, alpha=0.85, label="inscribed hexagon")
for name, line in side_lines.items():
    plot_line(ax, line, xlim=(-4.6, 4.6), color="#9ca3af")
plot_line(ax, pascal_line, xlim=(-4.6, 4.6), color=COLORS["red"], label="Pascal line")
plot_points(ax, P, labels=labels, color=COLORS["blue"], size=48)
plot_points(ax, X, labels=["AB meet DE", "BC meet EF", "CD meet FA"], color=COLORS["red"], size=54)
ax.set_xlim(-4.7, 4.7)
ax.set_ylim(-3.6, 4.6)
ax.legend(loc="upper right", fontsize=8)
pascal_fig = save_figure(fig, artifact("figures", "pascal_hexagon.png"))
display_artifact(notebook_relative(pascal_fig), width=760)

## Tangents, polars, and the dual use of the same matrix

The conic matrix does two jobs at once. It tests whether a point lies on the conic, and it converts points to lines by `P -> Q @ P`. When `P` lies on the conic, that line is the tangent at `P`. When `S` is outside the conic, `Q @ S` is the polar line of `S`. For the circle model, the polar of an exterior point cuts the circle at the two points where the tangents from `S` touch.

This view is useful because it makes dual statements cheap to test. If `T` is a contact point, the tangent line at `T` contains `S`, so `(Q @ T) @ S = 0`. The figure draws the exterior point, its polar chord, the two contact points, and the two tangents.

In [ ]:
def line_unit_circle_intersections(line: np.ndarray) -> np.ndarray:
    a, b, c = np.asarray(line, dtype=float)
    n2 = a * a + b * b
    closest = -c * np.array([a, b]) / n2
    direction = np.array([-b, a])
    direction = direction / np.linalg.norm(direction)
    radius_sq = 1.0 - float(closest @ closest)
    if radius_sq < -1e-12:
        return np.empty((0, 2))
    radius_sq = max(radius_sq, 0.0)
    offset = math.sqrt(radius_sq)
    return np.vstack([closest + offset * direction, closest - offset * direction])

S_h = np.array([1.85, 0.58, 1.0])
polar = polar_line(Q_circle, S_h)
contact = line_unit_circle_intersections(polar)
contact_h = np.column_stack([contact, np.ones(len(contact))])
tangent_lines = [polar_line(Q_circle, t) for t in contact_h]
polarity_errors = [abs(float(line @ S_h)) for line in tangent_lines]

fig, ax = new_axes(figsize=(7.4, 6.2), title="A point and its polar line with respect to a conic")
ax.plot(curve[:, 0], curve[:, 1], color=COLORS["ink"], linewidth=2.0, label="conic")
plot_line(ax, polar, xlim=(-1.4, 2.25), color=COLORS["orange"], label="polar of S")
for line in tangent_lines:
    plot_line(ax, line, xlim=(-1.4, 2.25), color=COLORS["teal"], label="tangent through S")
plot_points(ax, contact, labels=["T1", "T2"], color=COLORS["teal"], size=52)
S = affine(S_h)
ax.scatter([S[0]], [S[1]], s=70, c=COLORS["orange"], edgecolor="white", linewidth=0.9, zorder=6)
ax.text(S[0] + 0.05, S[1] + 0.05, "S", fontsize=10, color=COLORS["orange"])
for t in contact:
    ax.plot([S[0], t[0]], [S[1], t[1]], color=COLORS["teal"], linewidth=1.6, alpha=0.8)
ax.set_xlim(-1.35, 2.25)
ax.set_ylim(-1.25, 1.35)
handles, names = ax.get_legend_handles_labels()
unique = dict(zip(names, handles))
ax.legend(unique.values(), unique.keys(), loc="lower left", fontsize=8)
polarity_fig = save_figure(fig, artifact("figures", "polarity_tangent_chord.png"))
display_artifact(notebook_relative(polarity_fig), width=740)

## Intersections and the visible part of Bezout

Over the complex projective plane, two conics meet in four points when multiplicity is counted and the conics share no component. An affine real drawing may show four, two, one, or zero real points, but the projective count is the stable object.

The next example chooses two conics whose four intersections are all visible: the unit circle `x^2 + y^2 = 1` and the rectangular hyperbola `xy = 1/4`. The formula with `s = x + y` and `t = x - y` gives

`s^2 = 1 + 2a`, `t^2 = 1 - 2a`

for `a = 1/4`, producing four real solutions. The plot is paired with residual checks against both conic equations.

In [ ]:
alpha = 0.25
Q_hyperbola = np.array([[0.0, 0.5, 0.0], [0.5, 0.0, 0.0], [0.0, 0.0, -alpha]])
s_abs = math.sqrt(1.0 + 2.0 * alpha)
t_abs = math.sqrt(1.0 - 2.0 * alpha)
bezout_points = []
for s_sign in [-1.0, 1.0]:
    for t_sign in [-1.0, 1.0]:
        s = s_sign * s_abs
        t = t_sign * t_abs
        bezout_points.append([(s + t) / 2.0, (s - t) / 2.0])
bezout_points = np.array(bezout_points)
bezout_h = np.column_stack([bezout_points, np.ones(len(bezout_points))])
bezout_residuals = {
    "circle": [abs(hconic_value(Q_circle, p)) for p in bezout_h],
    "hyperbola": [abs(hconic_value(Q_hyperbola, p)) for p in bezout_h],
}

x = np.linspace(-1.35, 1.35, 500)
y = np.linspace(-1.35, 1.35, 500)
Xg, Yg = np.meshgrid(x, y)
F_circle = Xg * Xg + Yg * Yg - 1.0
F_hyperbola = Xg * Yg - alpha

fig, ax = new_axes(figsize=(7.2, 6.2), title="Two conics meeting in four real points")
ax.contour(Xg, Yg, F_circle, levels=[0], colors=[COLORS["ink"]], linewidths=2.0)
ax.contour(Xg, Yg, F_hyperbola, levels=[0], colors=[COLORS["purple"]], linewidths=2.0)
plot_points(ax, bezout_points, labels=["1", "2", "3", "4"], color=COLORS["red"], size=54)
ax.text(-1.25, 1.12, "x^2 + y^2 = 1", fontsize=9, color=COLORS["ink"])
ax.text(-1.25, 0.96, "xy = 1/4", fontsize=9, color=COLORS["purple"])
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.35, 1.35)
bezout_fig = save_figure(fig, artifact("figures", "bezout_four_intersections.png"))
display_artifact(notebook_relative(bezout_fig), width=720)

## Pencils and singular members

A pencil `Q0 + lambda Q1` is a projective line inside the vector space of quadratic forms. It packages a family of conics through a common intersection scheme. The determinant detects singular members: when `det(Q0 + lambda Q1) = 0`, the quadratic form has rank below three and the conic degenerates.

The HTML artifact below is an interactive contour view of one pencil. Moving the slider changes `lambda`; the determinant curve in the saved sanity data records where the singular members occur. This is a compact way to connect the algebraic phrase "pencil of conics" with the geometric act of watching a conic pass through degeneracy.

In [ ]:
def conic_field(Q: np.ndarray, X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    return Q[0, 0] * X * X + 2 * Q[0, 1] * X * Y + 2 * Q[0, 2] * X + Q[1, 1] * Y * Y + 2 * Q[1, 2] * Y + Q[2, 2]

lambda_values = np.linspace(-2.2, 2.2, 45)
determinants = np.array([np.linalg.det(Q_circle + lam * Q_hyperbola) for lam in lambda_values])
det_coeff = np.polyfit(lambda_values, determinants, deg=3)
det_roots = np.roots(det_coeff)

plot_x = np.linspace(-2.0, 2.0, 220)
plot_y = np.linspace(-2.0, 2.0, 220)
PX, PY = np.meshgrid(plot_x, plot_y)
frames = []
for lam in lambda_values:
    Q = Q_circle + lam * Q_hyperbola
    frames.append(
        go.Frame(
            data=[
                go.Contour(
                    x=plot_x,
                    y=plot_y,
                    z=conic_field(Q, PX, PY),
                    contours=dict(start=0, end=0, size=0.05, coloring="lines"),
                    line=dict(color="#2f6fed", width=3),
                    showscale=False,
                    name=f"lambda={lam:.2f}",
                )
            ],
            name=f"{lam:.2f}",
        )
    )

initial_Q = Q_circle + lambda_values[0] * Q_hyperbola
pencil_fig = go.Figure(
    data=[
        go.Contour(
            x=plot_x,
            y=plot_y,
            z=conic_field(initial_Q, PX, PY),
            contours=dict(start=0, end=0, size=0.05, coloring="lines"),
            line=dict(color="#2f6fed", width=3),
            showscale=False,
        )
    ],
    frames=frames,
)
pencil_fig.update_layout(
    title="Conic pencil Q0 + lambda Q1",
    width=760,
    height=620,
    xaxis=dict(range=[-2, 2], scaleanchor="y", scaleratio=1, zeroline=False),
    yaxis=dict(range=[-2, 2], zeroline=False),
    margin=dict(l=30, r=30, t=60, b=60),
    sliders=[
        dict(
            active=0,
            currentvalue=dict(prefix="lambda = "),
            steps=[
                dict(method="animate", args=[[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}], label=frame.name)
                for frame in frames
            ],
        )
    ],
)
pencil_html = save_plotly_html(pencil_fig, artifact("plots", "conic_pencil_slider.html"))

determinant_rows = [
    {"lambda": float(lam), "determinant": float(det)}
    for lam, det in zip(lambda_values, determinants)
]
determinant_table = save_csv(determinant_rows, artifact("tables", "pencil_determinants.csv"))
display_artifact(notebook_relative(pencil_html), width=780, height=640)

## Applied lab: fit, move, and test a conic

A useful way to internalize the chapter is to reverse the workflow. Instead of starting with an equation, start with five points. A conic has six symmetric matrix entries up to scale, so five point equations determine a one-dimensional nullspace in ordinary cases. Once that conic is recovered, you can transform the points by a projective matrix, transform the conic matrix by the inverse-transpose rule, and test that the transformed points still lie on the transformed conic.

This lab is small, but it mirrors larger projective-vision workflows: fit an incidence object, change coordinates, and verify that the equation and the data move together.

In [ ]:
def conic_from_five_affine_points(points: np.ndarray) -> np.ndarray:
    pts = np.asarray(points, dtype=float)
    if pts.shape != (5, 2):
        raise ValueError("expected five affine points")
    A = []
    for x0, y0 in pts:
        A.append([x0 * x0, 2 * x0 * y0, 2 * x0, y0 * y0, 2 * y0, 1.0])
    _, _, vh = np.linalg.svd(np.asarray(A, dtype=float))
    a, b, c, d, e, f = vh[-1]
    Q = np.array([[a, b, c], [b, d, e], [c, e, f]], dtype=float)
    return Q / np.linalg.norm(Q)

def transform_conic(Q: np.ndarray, H: np.ndarray) -> np.ndarray:
    invH = np.linalg.inv(H)
    return invH.T @ Q @ invH

fit_u = np.array([-2.0, -0.85, 0.15, 1.1, 2.75])
fit_points = np.array([affine(hpoint_on_conic(u)) for u in fit_u])
Q_fit = conic_from_five_affine_points(fit_points)
fit_scale = np.sum(Q_fit * Q_circle) / np.sum(Q_fit * Q_fit)
fit_error = np.linalg.norm(fit_scale * Q_fit - Q_circle)

H = np.array([[1.0, 0.35, 0.15], [-0.22, 1.15, 0.08], [0.18, -0.10, 1.0]])
fit_h = np.column_stack([fit_points, np.ones(len(fit_points))])
moved_h = np.array([H @ p for p in fit_h])
Q_moved = transform_conic(Q_circle, H)
move_residuals = [abs(hconic_value(Q_moved, p)) for p in moved_h]

lab_rows = [
    {"check": "five_point_fit_error", "value": float(fit_error)},
    {"check": "max_transformed_point_residual", "value": float(max(move_residuals))},
]
for idx, residual in enumerate(move_residuals, start=1):
    lab_rows.append({"check": f"transformed_point_{idx}_residual", "value": float(residual)})
for idx, value in enumerate(Q_fit.ravel(), start=1):
    lab_rows.append({"check": f"normalized_fit_matrix_entry_{idx}", "value": float(value)})
lab_table = save_csv(lab_rows, artifact("tables", "five_point_fit_lab.csv"))
display(pd.DataFrame(lab_rows))

## Final sanity checks

The final cell asserts the core computational claims from the notebook: the parametrized points satisfy the conic equation, cross-ratio survives the fractional linear change, Pascal's three constructed points are collinear, the polarity tangents pass through the exterior point, the Bezout example satisfies both conic equations at four displayed points, the five-point fit recovers the conic up to scale, and every generated artifact exists with nontrivial size.

In [ ]:
generated_artifacts = [
    param_fig,
    cr_fig,
    pascal_fig,
    polarity_fig,
    bezout_fig,
    pencil_html,
    cr_table,
    determinant_table,
    lab_table,
]

final_sanity = {
    "chapter": CHAPTER,
    "parametrization_max_residual": float(param_residual),
    "cross_ratio_error": float(cross_ratio_error),
    "pascal_collinearity_residual": float(pascal_residual),
    "polarity_max_tangent_incidence_error": float(max(polarity_errors)),
    "bezout_max_circle_residual": float(max(bezout_residuals["circle"])),
    "bezout_max_hyperbola_residual": float(max(bezout_residuals["hyperbola"])),
    "five_point_fit_error": float(fit_error),
    "projective_move_max_residual": float(max(move_residuals)),
    "pencil_determinant_roots": [
        {"real": float(root.real), "imag": float(root.imag)} for root in det_roots
    ],
    "artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in generated_artifacts],
}

sanity_path = save_json(final_sanity, artifact("checks", "final_sanity.json"))
generated_artifacts.append(sanity_path)

assert final_sanity["parametrization_max_residual"] < 1e-12
assert final_sanity["cross_ratio_error"] < 1e-12
assert final_sanity["pascal_collinearity_residual"] < 1e-12
assert final_sanity["polarity_max_tangent_incidence_error"] < 1e-12
assert final_sanity["bezout_max_circle_residual"] < 1e-12
assert final_sanity["bezout_max_hyperbola_residual"] < 1e-12
assert final_sanity["five_point_fit_error"] < 1e-12
assert final_sanity["projective_move_max_residual"] < 1e-12
assert_artifacts(generated_artifacts, min_bytes=128)

print(json.dumps(final_sanity, indent=2))

## Takeaways

A projective conic is both a quadratic equation and a projective line in disguise. The quadratic matrix makes incidence, tangency, polarity, coordinate change, and singularity tests executable. The `P^1` parametrization makes cross-ratios and conic automorphisms one-dimensional.

Pascal's theorem becomes a clean numerical collinearity check because the projective plane is linear algebra over homogeneous coordinates. Bezout's theorem shows why intersection counts are more stable than any one real affine drawing. Pencils connect the two attitudes: a linear family of matrices produces a geometric family of conics, and the determinant marks the exceptional members where the family degenerates.

For computation, the safest habit is to keep the homogeneous object until the last moment. Dehomogenize only to draw a chart, and let residuals in homogeneous coordinates verify the projective statement.